# Predicting Body Fat Percentage Using Machine Learning

**Dataset:** NHANES 2011–2012  
**Target:** Total body fat percentage (`DXDTOPF`)  
**Task:** Supervised regression


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Load and Merge NHANES Data

In [ ]:
demo = pd.read_sas("../data/DEMO_G.xpt")
bmx = pd.read_sas("../data/BMX_G.xpt")
paq = pd.read_sas("../data/PAQ_G.xpt")
dxx = pd.read_sas("../data/DXX_G.xpt")
diet = pd.read_sas("../data/DR1TOT_G.xpt")

df = (
    demo.merge(bmx, on="SEQN", how="inner")
        .merge(paq, on="SEQN", how="inner")
        .merge(dxx, on="SEQN", how="inner")
        .merge(diet, on="SEQN", how="inner")
)

print("Merged shape:", df.shape)


## 3. Variable Selection and Cleaning

In [ ]:
selected_vars = [
    "SEQN","RIDAGEYR","RIAGENDR","BMXHT","BMXWT","BMXBMI","BMXWAIST",
    "PAQ650","PAQ665","PAD680","PAQ710","PAQ715",
    "DR1TKCAL","DR1TPROT","DR1TCARB","DR1TTFAT","DXDTOPF"
]

ml_df = df[selected_vars].copy()

paq_vars = ["PAQ650", "PAQ665", "PAQ710", "PAQ715"]
for col in paq_vars:
    ml_df[col] = ml_df[col].replace(
        [7, 9, 77, 99, 777, 999, 7777, 9999], pd.NA
    )

print("Missing values:")
print(ml_df.isnull().sum().sort_values(ascending=False))

ml_clean = ml_df.dropna().copy()
print("Final shape:", ml_clean.shape)


## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(ml_clean["DXDTOPF"], bins=30)
plt.title("Distribution of Total Body Fat Percentage")
plt.xlabel("Body Fat Percentage")
plt.ylabel("Frequency")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(ml_clean["BMXBMI"], ml_clean["DXDTOPF"], alpha=0.35)
plt.title("BMI vs Body Fat Percentage")
plt.xlabel("BMI")
plt.ylabel("Body Fat Percentage")
plt.show()


In [ ]:
sex_values = sorted(ml_clean["RIAGENDR"].unique())
box_data = [
    ml_clean.loc[ml_clean["RIAGENDR"] == v, "DXDTOPF"]
    for v in sex_values
]
plt.figure(figsize=(7,5))
plt.boxplot(box_data, labels=[str(v) for v in sex_values])
plt.title("Body Fat Percentage by Sex")
plt.xlabel("Sex")
plt.ylabel("Body Fat Percentage")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(ml_clean["BMXWAIST"], ml_clean["DXDTOPF"], alpha=0.35)
plt.title("Waist Circumference vs Body Fat Percentage")
plt.xlabel("Waist Circumference")
plt.ylabel("Body Fat Percentage")
plt.show()


In [ ]:
corr_vars = [
    "RIDAGEYR","BMXHT","BMXWT","BMXBMI","BMXWAIST",
    "PAQ650","PAQ665","PAD680","PAQ710","PAQ715",
    "DR1TKCAL","DR1TPROT","DR1TCARB","DR1TTFAT","DXDTOPF"
]

corr = ml_clean[corr_vars].corr()

plt.figure(figsize=(12,9))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7)
plt.title("Correlation Heatmap of Selected Variables")
plt.tight_layout()
plt.show()


## 5. Train-Test Split and Encoding

In [ ]:
X = ml_clean.drop(columns=["SEQN", "DXDTOPF"])
y = ml_clean["DXDTOPF"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

X_train = pd.get_dummies(X_train, columns=["RIAGENDR"], drop_first=True)
X_test = pd.get_dummies(X_test, columns=["RIAGENDR"], drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Training observations:", X_train.shape[0])
print("Testing observations:", X_test.shape[0])


## 6. Machine Learning Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(max_depth=5, random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    )
}

predictions = {}
fitted_models = {}
rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted_models[name] = model
    predictions[name] = pred
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

results = pd.DataFrame(rows)
results.round(3)


## 7. Random Forest Feature Importance

In [ ]:
rf_model = fitted_models["Random Forest"]

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

importance.head(15)


In [ ]:
plt.figure(figsize=(9,6))
top_imp = importance.head(15).iloc[::-1]
plt.barh(top_imp["Feature"], top_imp["Importance"])
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


## 8. Actual vs Predicted

In [ ]:
y_pred_rf = predictions["Random Forest"]

plt.figure(figsize=(7,6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)

lo, hi = y_test.min(), y_test.max()
plt.plot([lo, hi], [lo, hi], linestyle="--")

plt.xlabel("Actual Body Fat Percentage")
plt.ylabel("Predicted Body Fat Percentage")
plt.title("Actual vs Predicted Body Fat Percentage - Random Forest")
plt.show()
